# Feature Selection for Fintech Complaint Risk Analysis

# Step 1. Notebook Overview

**Purpose of this Notebook :** The purpose of this notebook is to identify and retain the most informative features from the engineered feature set, reduce noise and dimensionality, improve model performance, and enhance interpretability.

**Why Feature Selection is Important**

* Reduces overfitting
* Improves training speed
* Enhances model generalization
* Makes model explanations clearer

Feature selection helps retain only meaningful signals while removing noise and redundant features.

# Step 2. Import Required Libraries

**Purpose :**   

1.Load only libraries required for selection techniques

2.Keep the notebook minimal and clean

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier


##### I use statistical and model-based feature selection techniques.

# Step 3. Load Feature-Engineered Dataset

**Purpose :**

1.Load model-ready numerical features

2.Avoid recomputation

In [2]:
file_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\data\processed\complaints_features.csv"
df = pd.read_csv(file_path)

df.shape


(3324, 308)

##### This dataset already contains engineered numerical features suitable for ML models.

In [3]:
df

,char_length,word_length,stopword_ratio,risk_keyword_flag,product_encoded,issue_encoded,complaint_year,complaint_month,accordance,account,...,xxxx balance,xxxx date,xxxx xxxx,xxxx xxxxxxxx,xxxxxxxx,xxxxxxxx balance,xxxxxxxx xxxx,xxxxyear,year,yet
0,57,8,0.0,0,6,32,2025,10,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
1,82,14,0.0,1,6,32,2025,10,0.555462,0.200713,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
2,26,5,0.0,0,7,32,2020,5,0.000000,1.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
3,364,54,0.0,1,6,32,2025,10,0.000000,0.169315,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
4,1599,233,0.0,1,6,32,2025,11,0.000000,0.112413,...,0.0,0.0,0.409834,0.040575,0.032309,0.0,0.039342,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3319,339,49,0.0,0,6,32,2025,11,0.000000,0.000000,...,0.0,0.0,0.117449,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
3320,1513,279,0.0,1,10,77,2025,9,0.000000,0.121353,...,0.0,0.0,0.235962,0.000000,0.041854,0.0,0.000000,0.0,0.000000,0.064745
3321,257,39,0.0,1,8,4,2025,9,0.000000,0.253319,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
3322,737,107,0.0,0,6,32,2025,11,0.245237,0.088615,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000


# Step 4. Create a Target Variable

**Purpose :**

1.Feature selection methods require a target

2.Since this is a self-project, we create a proxy target using risk keywords

In [4]:
df['target_risk'] = (df['risk_keyword_flag'] > 0).astype(int)

df['target_risk'].value_counts()

target_risk
1    2084
0    1240
Name: count, dtype: int64

##### In absence of labeled sentiment, I used a proxy risk target to guide feature selection.

# Step 5. Separate Features and Target

**Purpose :**

1.Clean separation of inputs and output

2.Prevent data leakage

In [5]:
X = df.drop(columns=['target_risk'])
y = df['target_risk']


##### Separating features and target avoids leakage during feature selection.

# Step 6. Remove Low-Variance Features

**Purpose :**

1.Features with near-zero variance add no predictive value

2.Common in high-dimensional TF-IDF features

In [6]:
variance_selector = VarianceThreshold(threshold=0.0001)
X_var = variance_selector.fit_transform(X)

selected_variance_features = X.columns[variance_selector.get_support()]

X_var_df = pd.DataFrame(X_var, columns=selected_variance_features)

X_var_df.shape


(3324, 307)

##### Low-variance features rarely contribute to model predictions, so I removed them early.

# Step 7. Train-Test Split (For Model-Based Selection)

**Purpose :**

1.Prevent selection bias

2.Simulate real training behavior

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_var_df, y, test_size=0.2, random_state=42, stratify=y
)

##### I perform feature selection only on training data to avoid leakage.

# Step 8. Statistical Feature Selection (Chi-Square)

**Purpose :**

1.Identify features statistically related to the target

2.Very effective for TF-IDF features

In [8]:
chi_selector = SelectKBest(score_func=chi2, k=50)
X_chi = chi_selector.fit_transform(X_train, y_train)

chi_features = X_train.columns[chi_selector.get_support()]

X_chi_df = pd.DataFrame(X_chi, columns=chi_features)

##### Chi-square identifies features most correlated with the target variable.

# Step 9. Model-Based Feature Importance (Random Forest)

**Purpose :**

1.Capture non-linear relationships

2.Rank features based on predictive contribution

In [9]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

importances = rf.feature_importances_

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

top_rf_features = importance_df.head(30)['feature'].tolist()


##### Tree-based models provide reliable feature importance scores.

# Step 10. Combine Selected Features

**Purpose :**

1.Use intersection of statistical + model-based selection

2.Reduce overfitting risk

In [10]:
final_features = list(set(chi_features).union(set(top_rf_features)))

X_selected = X_var_df[final_features]

X_selected.shape

(3324, 56)

##### Combining multiple selection techniques improves robustness.

# Step 11. Save Selected Features Dataset

**Purpose :**

1.Persist final selected features

2.Enable clean model training

In [11]:
final_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\data\processed\complaints_selected_features.csv"

X_selected['target_risk'] = y.values
X_selected.to_csv(final_path, index=False)

print("Feature-selected dataset saved successfully")

C:\Users\hp\AppData\Local\Temp\ipykernel_6112\3415493642.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_selected['target_risk'] = y.values


Feature-selected dataset saved successfully


##### I save the selected features separately to keep the pipeline modular.